In [383]:
from sympy import Function, Symbol, symbols, simplify, Eq, Symbol, latex, pprint, collect, expand
from sympy import init_printing
from IPython.display import display, Math

init_printing(use_latex=True)

In [384]:
import re
from IPython.display import display, Math

def color_terms(latex_str):
    latex_str = re.sub(
        r'(q_\{(\d+)\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{orange} \1{\\left(\3\\right)}}',
        latex_str
    )
    latex_str = re.sub(
        r'(\\epsilon_\{([A-Za-z])\})\{\\left\((.+?)\\right\)\}',
        r'{\\color{skyblue} \1{\\left(\3\\right)}}',
        latex_str
    )
    return latex_str

In [367]:
# ── time variable ─────────────────────────────────────────────────────────────
t = Symbol('t')

# ── delay parameters ──────────────────────────────────────────────────────────
tau12, tau21, tau13, tau31, tau23, tau32 = symbols(
    r'\tau_{12} \tau_{21} \tau_{13} \tau_{31} \tau_{23} \tau_{32}',
    real=True, positive=True
)

# ── laser angular frequencies (physical + measurement offsets) ────────────────
omega1, omega2, omega3 = symbols(r'\omega_1 \omega_2 \omega_3', real=True)
omega1m, omega2m, omega3m = symbols(r'\omega_1^m \omega_2^m \omega_3^m', real=True)

In [368]:
# ── abstract time-dependent functions ─────────────────────────────────────────
phi1  = Function(r'\phi_1')   # laser phase noise, s/c 1
phi2  = Function(r'\phi_2')
phi3  = Function(r'\phi_3')

epsilonA = Function(r'\epsilon_A')  # clock noise, s/c 1 (master)
epsilonB = Function(r'\epsilon_B')  # clock noise, s/c 2
epsilonC = Function(r'\epsilon_C')  # clock noise, s/c 3

omegaDDSA, omegaDDSB, omegaDDSC = symbols(r'\omega^{DDS}_A \omega^{DDS}_B \omega^{DDS}_C', real=True)

q1 = Function('q_1')
q2 = Function('q_2')
q3 = Function('q_3')

In [369]:
def D(expr, tau):
    return expr.subs(t, t - tau)

In [ ]:
# Map spacecraft index → (phi, q, omega, omega_m, clock_epsilon)
sc = {
    1: (phi1, q1, omega1, omega1m, epsilonA, omegaDDSA),
    2: (phi2, q2, omega2, omega2m, epsilonB, omegaDDSB),
    3: (phi3, q3, omega3, omega3m, epsilonC, omegaDDSC),
}

tau = {
    (1,2): tau12, (2,1): tau21,
    (1,3): tau13, (3,1): tau31,
    (2,3): tau23, (3,2): tau32,
}

eta    = {}
etaSB  = {}
etaU  = {} # unmixed
DDS  = {}
r = {}

include_phi = True 
include_clock_noise = True
include_board_jitter = True  

for (i, j) in tau:
    phi_i, q_i, om_i, omm_i, eps_i, omegaDDSi = sc[i]
    phi_j, q_j, om_j, omm_j, eps_j, omegaDDSj = sc[j]
    t_ij = tau[(i, j)]

    phi_terms = (D(phi_j(t), t_ij) - phi_i(t)) if include_phi else 0

    clock_terms_C = (- (om_j - om_i) * q_i(t)) if include_clock_noise else 0
    clock_terms_SB = (- (om_j - om_i + omm_j - omm_i) * q_i(t) - omm_i * q_i(t) + omm_j * D(q_j(t), t_ij)) if include_clock_noise else 0
    
    board_terms_c = (om_j * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0
    board_terms_SB = ((om_j + omm_j) * (eps_i(t) - D(eps_i(t), t_ij))) if include_board_jitter else 0

    eta[(i,j)] = (phi_terms + clock_terms_C + board_terms_c)

    etaSB[(i,j)] = collect(expand(phi_terms + clock_terms_SB + board_terms_SB), [q_i(t), q_j(t)])

    r[(i,j)] = simplify((eta[(i,j)] - etaSB[(i,j)]) / omm_j)

    etaU[(i,j)] = (D(phi_j(t), t_ij) - om_j  * q_i(t) + (om_j * (eps_i(t) - D(eps_i(t), t_ij))))
    
    DDS[(i,j)] = ( - omegaDDSi  * q_i(t) + omegaDDSi * eps_i(t) ) # DDS measured by the other PM

In [ ]:
if True:
    for (i, j) in tau:
        display(Math(r'\eta_{' + str(i) + str(j) + '} = ' + latex(eta[(i,j)])))
        #display(Math(r'\eta_{' + str(i) + str(j) + '}^{U} = ' + latex(etaU[(i,j)])))
        display(Math(r'\eta_{' + str(i) + str(j) + '}^{SB} = ' + latex(etaSB[(i,j)])))
        display(Math(r'r_{' + str(i) + str(j) + '} = ' + latex(r[(i,j)])))
        display(Math(r'DDS_{' + str(i) + str(j) + '} = ' + latex(DDS[(i,j)])))
        print("------------------------------------------------------")

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

------------------------------------------------------


We will now apply the Vallisneri's algorithm to the clock jitters following the article. For generation 2 TDI Michelson type combinations we write the delay polynomials as

$$
\begin{align}
P_{12} &= -\left(1-\mathbf{D}_{131}-\mathbf{D}_{13121}+\mathbf{D}_{1213131}\right), \\
P_{23} &= 0, \\
P_{31} &= \left(1-\mathbf{D}_{121}-\mathbf{D}_{12131}+\mathbf{D}_{1312121}\right)\mathbf{D}_{13}, \\
P_{21} &= -\left(1-\mathbf{D}_{131}-\mathbf{D}_{13121}+\mathbf{D}_{1213131}\right)\mathbf{D}_{12}, \\
P_{32} &= 0, \\
P_{13} &= \left(1-\mathbf{D}_{121}-\mathbf{D}_{12131}+\mathbf{D}_{1312121}\right).
\end{align}
$$

Now we start constructing the variables. Let's look at $R_{12}$. First, the unity term, we skip it, but remember it corresponds to a $ q_1$ term. For the other delay terms, we write for each
\begin{alignat*}{3}
    \mathbf{D}_{131}: \qquad
    & \mathbf{D}_{131} q_1 -  q_1
    & {}={}& r_{13} + \mathbf{D}_{13} r_{31}, \\[3pt]
    \mathbf{D}_{13121}: \qquad
    & \mathbf{D}_{13121} q_1 -  q_1
    & {}={}& r_{13} + \mathbf{D}_{13} r_{31}
            + \mathbf{D}_{131} r_{12}
            + \mathbf{D}_{1312} r_{21}, \\[3pt]
    \mathbf{D}_{1213131}: \qquad
    & \mathbf{D}_{1213131}\dot q_1 - \dot q_1
    & {}={}& r_{12} + \mathbf{D}_{12} r_{21}
            + \mathbf{D}_{121} r_{13}
            + \mathbf{D}_{1213} r_{31}
            + \mathbf{D}_{12131} r_{12}
            + \mathbf{D}_{121313} r_{31}.
\end{alignat*}
Now we plug all of these into $R_{12}$, remembering to include the $\dot q_1$ from the identity term
\begin{align*}
    R_{12}= &\dot q_1+\mathbf{D}_{131}\dot q_1 - \dot q_1+\mathbf{D}_{13121}\dot q_1 - \dot q_1-\mathbf{D}_{1213131}\dot q_1 + \dot q_1=\\
     &r_{13} + \mathbf{D}_{13} r_{31} \\
     +&r_{13} + \mathbf{D}_{13} r_{31} + \mathbf{D}_{131} r_{12} + \mathbf{D}_{1312} r_{21}
     \\-&r_{12} - \mathbf{D}_{12} r_{21} - \mathbf{D}_{121} r_{13} - \mathbf{D}_{1213} r_{31} - \mathbf{D}_{12131} r_{12} - \mathbf{D}_{121313} r_{31}=\\
     &\left(2-\mathbf{D}_{121}-\mathbf{D}_{12131}\right)\left(r_{13}+\mathbf{D}_{13} r_{31}\right)+\left(\mathbf{D}_{131}-1\right)\left( r_{12}+\mathbf{D}_{12} r_{21}\right).
\end{align*}

And from the article we can copy the following terms
$$
\begin{align}
R_{12} &= -(1 - \mathbf{D}_{131})(r_{12} + \mathbf{D}_{12}r_{21})
        + (2 - \mathbf{D}_{121} - \mathbf{D}_{12131})(r_{13} + \mathbf{D}_{13}r_{31}), \\
R_{23} &= 0, \\
R_{31} &= -(2 - \mathbf{D}_{131} - \mathbf{D}_{13121})(r_{12} + \mathbf{D}_{12}r_{21})
        + (1 - \mathbf{D}_{121})(r_{13} + \mathbf{D}_{13}r_{31}) \notag \\
       &\quad + (1 - \mathbf{D}_{121} - \mathbf{D}_{12131} + \mathbf{D}_{1312121})r_{13}, \\
R_{21} &= -(1 - \mathbf{D}_{131})(r_{12} + \mathbf{D}_{12}r_{21})
        - (1 - \mathbf{D}_{131} - \mathbf{D}_{13121} + \mathbf{D}_{1213131})r_{12} \notag \\
       &\quad + (2 - \mathbf{D}_{121} - \mathbf{D}_{12131})(r_{13} + \mathbf{D}_{13}r_{31}), \\
R_{32} &= 0, \\
R_{13} &= -(2 - \mathbf{D}_{131} - \mathbf{D}_{13121})(r_{12} + \mathbf{D}_{12}r_{21})
        + (1 - \mathbf{D}_{121})(r_{13} + \mathbf{D}_{13}r_{31}).
\end{align}
$$

We actually look at first generation TDI here. I do not know how to properly cancel out the timechaning delays with symbolic calculations.

In [372]:
def P12(expr): return expr - D(expr, tau13 + tau31)
def P21(expr): return D(expr, tau12) - D(expr, tau12 + tau13 + tau31)
def P13(expr): return -(expr - D(expr, tau12 + tau21))
def P31(expr): return -(D(expr, tau13) - D(expr, tau13 + tau12 + tau21))

# X1
X1 = collect(expand(P13(eta[(1,3)] + D(eta[(3,1)], tau13))+ P12(eta[(1,2)] + D(eta[(2,1)], tau12))), [q1(t), q2(t), q3(t)])

In [373]:
display(Math(r'X_1 = ' + latex(X1)))

<IPython.core.display.Math object>

In [374]:
R = {}

R[(1,2)] = -(r[(1,3)] + D(r[(3,1)], tau13))

R[(1,3)] = r[(1,2)] + D(r[(2,1)], tau12)

R[(2,1)] = (r[(1,2)]
             - r[(1,3)]
             - D(r[(3,1)], tau13)
             - D(r[(1,2)], tau13 + tau31))

R[(3,1)] = (-r[(1,3)]
              + r[(1,2)]
              + D(r[(2,1)], tau12)
              + D(r[(1,3)], tau12 + tau21))

R[(2,3)] = 0
R[(3,2)] = 0

In [375]:
a = {
    (1,2): omega1 - omega2,
    (2,1): omega2 - omega1,
    (1,3): omega1 - omega3,
    (3,1): omega3 - omega1,
    (2,3): omega2 - omega3,
    (3,2): omega3 - omega2,
}

# clockwise triplets I+_3
triplets = [(1,2,3), (2,3,1), (3,1,2)]

correction = 0
for (i, j, k) in triplets:
    correction -= (
        - a[(i,j)] * R[(i,j)]
        - a[(i,k)] * R[(i,k)]
    )

X1c = simplify(X1 + correction)

In [376]:
print("The X1 expression is:")
display(Math(r'X_1 = ' + latex(X1)))
print("The clock-jitter correction term is:")
display(Math(r'X_1 = ' + latex(simplify(-correction))))
print("The corrected X1 expression is:")
display(Math(r'X_1 = ' + latex(X1c)))

The X1 expression is:


<IPython.core.display.Math object>

The clock-jitter correction term is:


<IPython.core.display.Math object>

The corrected X1 expression is:


<IPython.core.display.Math object>

In [377]:
x1cc = collect(X1c/omega1, [epsilonA(t), epsilonB(t), epsilonC(t)])
display(Math(latex(x1cc)))

<IPython.core.display.Math object>

In [290]:
check = simplify(r[(1,2)] + DDS[(1,2)]/sc[1][5] - D(DDS[(1,3)]/sc[1][5], tau12))
display(Math(r'r_{12}^c = ' + latex(check)))

<IPython.core.display.Math object>

In [ ]:
# epsilon assigned by shared board (link pair)
"""
board = {
    (1,2): epsilonA, (2,1): epsilonA,
    (2,3): epsilonB, (3,2): epsilonB,
    (1,3): epsilonC, (3,1): epsilonC,
}

for (i, j) in tau:
    phi_i, q_i, om_i, omm_i, eps_i = sc[i]
    phi_j, q_j, om_j, omm_j, eps_j = sc[j]
    t_ij = tau[(i, j)]
    eps_board = board[(i,j)]   # shared board epsilon

    phi_terms = (D(phi_j(t), t_ij) - phi_i(t)) if include_phi else 0

    clock_terms_C  = (-(om_j - om_i) * q_i(t)) if include_clock_noise else 0
    clock_terms_SB = (-(om_j - om_i + omm_j - omm_i) * q_i(t)
                      - omm_i * q_i(t)
                      + omm_j * D(q_j(t), t_ij)) if include_clock_noise else 0

    board_terms_c  = (om_j  * (eps_board(t) - D(eps_board(t), t_ij))) if include_board_jitter else 0
    board_terms_SB = ((om_j + omm_j) * (eps_board(t) - D(eps_board(t), t_ij))) if include_board_jitter else 0

    eta[(i,j)]   = phi_terms + clock_terms_C  + board_terms_c
    etaSB[(i,j)] = collect(expand(phi_terms + clock_terms_SB + board_terms_SB), [q_i(t), q_j(t)])
    r[(i,j)]     = simplify((eta[(i,j)] - etaSB[(i,j)]) / omm_j)
"""

print("This is for when one board is realted to an arm rather than a spacecraft")


This is one board is realted to an arm rather than a spacecraft
